# LeetCode #68: Text Justification

https://leetcode.com/problems/text-justification/

## Comparison of Approaches

| Approach | Time | Space | Notes |
|----------|------|-------|-------|
| Brute Force (re-scan) | O(n²) | O(n) | Re-scan words for each line |
| **Greedy Line Packing ★** | **O(n)** | **O(n)** | Pack words greedily, then distribute spaces evenly |

---

## Understanding the Methods

### Brute Force
For each line, scan all remaining words from scratch to find how many fit. Redundant work because we revisit words already considered.

### Optimal: Greedy Line Packing ★
Use a single pointer to scan words. For each line:
1. **Pack**: greedily add words while `currentLength + 1 + nextWord.Length <= maxWidth`.
2. **Distribute Spaces**: compute `totalSpaces = maxWidth - sumOfWordLengths`. Distribute across `gaps = wordCount - 1` gaps. Each gap gets `totalSpaces / gaps` spaces; the first `totalSpaces % gaps` gaps get one extra (left-heavy distribution).
3. **Special Cases**: last line → left-justify (single spaces + right padding). Single word on a line → all spaces go right.

**Constraints:**
* 1 <= words.length <= 300
* 1 <= words[i].length <= 20
* Sum of all words[i].length <= 1000
* 1 <= maxWidth <= 100
* Each word fits within maxWidth

## Solutions

### C#

In [ ]:
public class Solution {
    public IList<string> FullJustify(string[] words, int maxWidth) {
        var result = new List<string>();
        int i = 0, n = words.Length;

        while (i < n) {
            // Greedy: pack words onto this line
            int lineChars = words[i].Length;
            int j = i + 1;
            while (j < n && lineChars + 1 + words[j].Length <= maxWidth) {
                lineChars += 1 + words[j].Length;
                j++;
            }
            // words[i..j-1] are on this line
            int wordCount = j - i;
            int totalWordLen = lineChars - (wordCount - 1); // sum of word lengths only
            int totalSpaceNeeded = maxWidth - totalWordLen;

            var sb = new System.Text.StringBuilder();
            sb.Append(words[i]);

            bool isLastLine = (j == n);
            if (isLastLine || wordCount == 1) {
                // Left-justify: single space between words, pad right
                for (int k = i + 1; k < j; k++) {
                    sb.Append(' ');
                    sb.Append(words[k]);
                }
                sb.Append(' ', maxWidth - sb.Length);
            } else {
                int gaps = wordCount - 1;
                int eachGap = totalSpaceNeeded / gaps;
                int extraSpaces = totalSpaceNeeded % gaps;
                for (int k = i + 1; k < j; k++) {
                    int spaces = eachGap + (k - i <= extraSpaces ? 1 : 0);
                    sb.Append(' ', spaces);
                    sb.Append(words[k]);
                }
            }

            result.Add(sb.ToString());
            i = j;
        }

        return result;
    }
}

### Python

In [ ]:
class Solution:
    def fullJustify(self, words: list[str], maxWidth: int) -> list[str]:
        result = []
        i, n = 0, len(words)

        while i < n:
            # Greedily pack words
            line_len = len(words[i])
            j = i + 1
            while j < n and line_len + 1 + len(words[j]) <= maxWidth:
                line_len += 1 + len(words[j])
                j += 1

            word_count = j - i
            total_word_len = sum(len(words[k]) for k in range(i, j))
            total_spaces = maxWidth - total_word_len

            is_last = (j == n)
            if is_last or word_count == 1:
                # Left-justify
                line = " ".join(words[i:j])
                line += " " * (maxWidth - len(line))
            else:
                gaps = word_count - 1
                each = total_spaces // gaps
                extra = total_spaces % gaps
                line = words[i]
                for k in range(1, word_count):
                    spaces = each + (1 if k <= extra else 0)
                    line += " " * spaces + words[i + k]

            result.append(line)
            i = j

        return result

### Go

In [ ]:
func fullJustify(words []string, maxWidth int) []string {
    result := []string{}
    n := len(words)
    i := 0

    for i < n {
        lineLen := len(words[i])
        j := i + 1
        for j < n && lineLen+1+len(words[j]) <= maxWidth {
            lineLen += 1 + len(words[j])
            j++
        }

        wordCount := j - i
        totalWordLen := 0
        for k := i; k < j; k++ {
            totalWordLen += len(words[k])
        }
        totalSpaces := maxWidth - totalWordLen

        var sb strings.Builder
        sb.WriteString(words[i])

        isLast := j == n
        if isLast || wordCount == 1 {
            for k := i + 1; k < j; k++ {
                sb.WriteByte(' ')
                sb.WriteString(words[k])
            }
            for sb.Len() < maxWidth {
                sb.WriteByte(' ')
            }
        } else {
            gaps := wordCount - 1
            each := totalSpaces / gaps
            extra := totalSpaces % gaps
            for k := 1; k < wordCount; k++ {
                spaces := each
                if k <= extra {
                    spaces++
                }
                for s := 0; s < spaces; s++ {
                    sb.WriteByte(' ')
                }
                sb.WriteString(words[i+k])
            }
        }

        result = append(result, sb.String())
        i = j
    }

    return result
}

### Rust

In [ ]:
impl Solution {
    pub fn full_justify(words: Vec<String>, max_width: i32) -> Vec<String> {
        let max_w = max_width as usize;
        let mut result = Vec::new();
        let n = words.len();
        let mut i = 0;

        while i < n {
            let mut line_len = words[i].len();
            let mut j = i + 1;
            while j < n && line_len + 1 + words[j].len() <= max_w {
                line_len += 1 + words[j].len();
                j += 1;
            }

            let word_count = j - i;
            let total_word_len: usize = words[i..j].iter().map(|w| w.len()).sum();
            let total_spaces = max_w - total_word_len;

            let is_last = j == n;
            let line = if is_last || word_count == 1 {
                let mut s = words[i..j].join(" ");
                while s.len() < max_w { s.push(' '); }
                s
            } else {
                let gaps = word_count - 1;
                let each = total_spaces / gaps;
                let extra = total_spaces % gaps;
                let mut s = words[i].clone();
                for k in 1..word_count {
                    let spaces = each + if k <= extra { 1 } else { 0 };
                    s.push_str(&" ".repeat(spaces));
                    s.push_str(&words[i + k]);
                }
                s
            };

            result.push(line);
            i = j;
        }

        result
    }
}

## Example Scenarios

### 1. Common Case — Even Distribution
**Input:** `words = ["What","must","be","acknowledgment","shall","be"]`, `maxWidth = 16`
**Line 1:** "What   must   be" (6 words → 2 gaps of 3 spaces each)
**Output:** Lines fully justified to exactly 16 chars.

### 2. Slightly Complex — Uneven Extra Spaces
**Input:** `words = ["a","b","c","d","e"]`, `maxWidth = 6`
First line packs "a b c": 3 words, 5 spaces to fill 2 gaps → gap1=3, gap2=2
**Output:** "a  b c" (left gap gets the extra)

### 3. Edge Case — Last Line Always Left-Justified
**Input:** Any input where the last line has multiple words.
Regardless of space count, last line uses single spaces + right padding.
**Output:** "hello world   " (not "hello   world ")

### 4. Edge Case — Single Word on a Line
**Input:** A word whose length equals maxWidth, e.g. `words=["pneumonoultramicroscopicsilicovolcanoconiosis"]`, `maxWidth=45`
No gaps exist; all padding goes to the right.
**Output:** The word followed by trailing spaces.

### 5. Almost-Impossible but Plausible — All Words Same Length
**Input:** `words=["aa","bb","cc","dd"]`, `maxWidth=5`
"aa bb" is 5 chars exactly, "cc dd" is 5 chars exactly → no extra spaces needed.
**Output:** ["aa bb", "cc dd"]

![image.png](attachment:image.png)